# [Scorio Math](https://huggingface.co/buckets/harimo/scorio-math)

Scorio Math contains 59,520 sampled attempts from four model configurations and five
competition-math benchmarks. Every question has 80 attempts per model. One Parquet file is
one candidate pool: one model, one question, and 80 rows ordered by seed.

Unlike Scorio Lite, this Bucket stores the top-20 candidate distribution at every prompt
and completion token position. Use it for token-level uncertainty signals. For accuracy,
ranking, or voting without these distributions, project only the columns you need.


## Check Storage Bucket dependencies

Storage Bucket loading requires `datasets>=5.0.0` and `huggingface_hub>=1.5.0`.
Run this cell before loading data. It installs only requirements that the active kernel
does not satisfy. Restart the kernel if the cell installs an update.


In [7]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

from packaging.version import Version

minimum_versions = {
    "datasets": Version("5.0.0"),
    "huggingface_hub": Version("1.5.0"),
}

installed_versions = {}
requirements = []
for package, minimum in minimum_versions.items():
    try:
        installed = Version(version(package))
    except PackageNotFoundError:
        installed = None
    installed_versions[package] = installed
    if installed is None or installed < minimum:
        requirements.append(f"{package}>={minimum}")

if requirements:
    print("Installing:", ", ".join(requirements))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *requirements]
    )
    print("Restart the kernel, then run the notebook from the top.")
else:
    for package, minimum in minimum_versions.items():
        print(f"{package}: {installed_versions[package]} (required >= {minimum})")


datasets: 5.0.1 (required >= 5.0.0)
huggingface_hub: 1.29.0 (required >= 1.5.0)


## Load one candidate pool with `datasets`

Storage Buckets use `buckets/<owner>/<name>` and an explicit `data_files` selection. This
example limits the load to one small `gpt-oss-20b_low` question file.


In [8]:
from collections import Counter

import pandas as pd
import pyarrow.parquet as pq
from datasets import load_dataset
from IPython.display import display

from scorio import agg

bucket_name = "buckets/harimo/scorio-math"
model_name = "gpt-oss-20b_low"
task = "aime_2026"
question_id = 0

data_files = {task: f"data/{model_name}/{task}/q{question_id:02d}.parquet"}
stream = load_dataset(bucket_name, data_files=data_files, split=task, streaming=True)
first = next(iter(stream))

print(first["task"], first["model_key"], first["data_id"], first["seed"])
print("columns:", len(first))
print("token fields:", sorted(first["tokens"]))


aime_2026 gpt-oss-20b_low 0 0
columns: 47
token fields: ['completion_avg_logprob', 'completion_logprob_list', 'completion_ppl', 'completion_rank_list', 'completion_sum_logprob', 'completion_token_list', 'completion_topk_logprobs_list', 'prompt_avg_logprob', 'prompt_logprob_list', 'prompt_ppl', 'prompt_rank_list', 'prompt_sum_logprob', 'prompt_token_list', 'prompt_topk_logprobs_list']


## Inspect the complete 80-attempt pool

The low-reasoning file is small enough to read in full. Other models can have much larger
rows; use streaming or column projection when the top-20 lists are not needed.


In [9]:
path = (
    "hf://buckets/harimo/scorio-math/"
    f"data/{model_name}/{task}/q{question_id:02d}.parquet"
)
pool = pq.read_table(path).to_pylist()

assert len(pool) == 80
assert [row["seed"] for row in pool] == list(range(80))

print("attempts:", len(pool))
print("ground truth:", pool[0]["ground_truth"])
print("five most common answers:", Counter(row["extracted_answer"] for row in pool).most_common(5))
print("rule-based accuracy:", sum(row["evalscope_is_correct"] for row in pool) / len(pool))


attempts: 80
ground truth: 277
five most common answers: [('277', 77), ('221', 2), ('37', 1)]
rule-based accuracy: 0.9625


## One attempt and its top-20 distributions

Each candidate entry has `token`, `token_id`, `logprob`, and `rank`. Completion rows place
the sampled token at index 0; that entry's `rank` is positional, not its vocabulary rank.
Use `completion_rank_list` for the sampled token's vocabulary rank.


In [10]:
attempt = pool[0]
tokens = attempt["tokens"]
position = 12
candidates = tokens["completion_topk_logprobs_list"][position]

display(pd.DataFrame(candidates[:8]))
print("sampled token:", tokens["completion_token_list"][position])
print("sampled-token vocabulary rank:", tokens["completion_rank_list"][position])
print("top-k index-0 rank field:", candidates[0]["rank"])
print("completion positions:", len(tokens["completion_topk_logprobs_list"]))


,token,token_id,logprob,rank
0,p,275,-0.000078,1
1,t,260,-9.500078,2
2,T,353,-13.250078,3
3,p,79,-14.000078,4
4,,220,-14.500078,5
5,(,350,-16.250078,6
6,r,428,-16.750078,7
7,P,398,-17.250078,8


sampled token:  p
sampled-token vocabulary rank: 1
top-k index-0 rank field: 1
completion positions: 1073


## Confidence signals from `scorio.aggregate`

Scorio confidence functions take numeric top-k log-probabilities, so the code extracts the
`logprob` value from each candidate struct. Entropy and varentropy are uncertainties; the
other columns below increase with confidence.


In [11]:
def topk_logprobs(record):
    return [
        [candidate["logprob"] for candidate in position]
        for position in record["tokens"]["completion_topk_logprobs_list"]
    ]


confidence = []
for record in pool[:3]:
    topk = topk_logprobs(record)
    confidence.append({
        "seed": record["seed"],
        "self_certainty": agg.self_certainty(topk),
        "deepconf": agg.deepconf_confidence(topk),
        "entropy": agg.token_entropy(topk),
        "varentropy": agg.varentropy(topk),
        "max_probability": agg.max_softmax_probability(topk),
        "logprob_margin": agg.logprob_margin(topk),
    })

display(pd.DataFrame(confidence).round(4))


,seed,self_certainty,deepconf,entropy,varentropy,max_probability,logprob_margin
0,0,12.5181,15.5141,0.2020,0.2448,0.9263,9.1336
1,1,12.3201,15.3164,0.2314,0.2671,0.9145,8.6585
2,2,12.8419,15.8378,0.1917,0.2300,0.9303,9.5222


## Read light columns only

PyArrow projects columns at read time. Use column projection for metrics that do not need
the top-20 distributions.


In [12]:
light = pq.read_table(
    path,
    columns=["data_id", "seed", "evalscope_is_correct", "finish_reason", "num_completion_tokens"],
).to_pandas()

print(light.shape)
display(light.head())


(80, 5)


,data_id,seed,evalscope_is_correct,finish_reason,num_completion_tokens
0,0,0,1,stop,1073
1,0,1,1,stop,1055
2,0,2,1,stop,1115
3,0,3,1,stop,1046
4,0,4,1,stop,1033
